# Exploratory Data Analysis

In [9]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import pandas as pd
import plotly.express as px
from tqdm import tqdm

from solar_predictor.storage import find_files_at_loc, get_location_prefix, read_csv

MONTHS = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
DATETIME_FORMAT = "%Y-%m-%d %H:%M"

# Read Data

In [11]:
loc_lat, loc_lon = 42.0, 21.43

csvs = find_files_at_loc(get_location_prefix(loc_lat, loc_lon), "/sarah3.csv")
dfs = [read_csv(csv) for csv in tqdm(csvs, desc="Reading CSVs")]
raw_df = pd.concat(dfs, ignore_index=True)

Reading CSVs: 100%|██████████| 37/37 [04:09<00:00,  6.75s/it]


In [38]:
df = raw_df.copy()

df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
df = df.set_index("datetime")

df.head()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
1990-01-01 00:00:00+00:00,631152000,0.0,0.0,0.0,0.0
1990-01-01 00:30:00+00:00,631153800,0.0,0.0,0.0,0.0
1990-01-01 01:00:00+00:00,631155600,0.0,0.0,0.0,0.0
1990-01-01 01:30:00+00:00,631157400,0.0,0.0,0.0,0.0
1990-01-01 02:00:00+00:00,631159200,0.0,0.0,0.0,0.0


# Data Quality

## Basic Checks

In [ ]:
df.describe()


,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
count,6.455520e+05,641974.000000,641962.000000,641962.000000,641962.000000
mean,1.212148e+09,175.828710,107.919140,68.331826,185.687606
std,3.354389e+08,266.189507,202.574334,94.506365,294.126137
min,6.311520e+08,0.000000,0.000000,0.000000,0.000000
25%,9.216500e+08,0.000000,0.000000,0.000000,0.000000
50%,1.212148e+09,4.000000,0.000000,2.000000,0.000000
75%,1.502646e+09,279.000000,106.000000,127.000000,320.900000
max,1.793144e+09,1044.000000,925.000000,483.000000,1029.500000


There are no negative values, the maximum values are within the expected range, and some missing values still need to be handled.

## Missing and Duplicated Timestamps

In [28]:
expected = pd.date_range(df.index.min(), df.index.max(), freq="30min")
missing = expected.difference(df.index)

print(f"Expected: {len(expected)}, actual: {len(df)}, missing: {len(missing)}")
print(f"Duplicated timestamps: {df.index.duplicated().sum()}")

Expected: 645648, actual: 645648, missing: 0
Duplicated timestamps: 0


There are no missing or duplicated timestamps.

## Missing Values
While all timestamps are present, they may contain missing values in them.

In [29]:
1 - df.count() / df.shape[0]

timestamp                   0.000000
shortwave_radiation         0.005690
direct_radiation            0.005709
diffuse_radiation           0.005709
direct_normal_irradiance    0.005709
dtype: float64

There are missing values withing the rows with valid timestamps.

### Count per Year

In [30]:
missing_count = df.isna().groupby(df.index.year.rename("year")).sum().reset_index()
px.line(
    missing_count,
    x="year",
    y="shortwave_radiation",
    title="Number of missing periods per year",
)

The average year has about 100 missing values, except three years: 1990 with 400, 2009 with 380 and 2026 with about 530.
The missigness in 1990 and 2009 is probably of technical reasons while the data in 2026 may be still processing.

### Missing Values in the Tail
The satellite needs time to send the data to planet Earth, remove the last rows since they don't contain data.

In [31]:
df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
2026-10-29 21:30:00+00:00,1793309400,NaN,NaN,NaN,NaN
2026-10-29 22:00:00+00:00,1793311200,NaN,NaN,NaN,NaN
2026-10-29 22:30:00+00:00,1793313000,NaN,NaN,NaN,NaN
2026-10-29 23:00:00+00:00,1793314800,NaN,NaN,NaN,NaN
2026-10-29 23:30:00+00:00,1793316600,NaN,NaN,NaN,NaN


#### Filter

In [40]:
last_complete_row = df[(~df.isna()).all(axis=1)].iloc[-1]

filter = df["timestamp"] <= last_complete_row["timestamp"]

print(f"Filtering {len(df) - filter.sum()} items.")
df = df.loc[filter]

df.tail()

Filtering 0 items.


,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
2026-10-27 21:30:00+00:00,1793136600,0.0,0.0,0.0,0.0
2026-10-27 22:00:00+00:00,1793138400,0.0,0.0,0.0,0.0
2026-10-27 22:30:00+00:00,1793140200,0.0,0.0,0.0,0.0
2026-10-27 23:00:00+00:00,1793142000,0.0,0.0,0.0,0.0
2026-10-27 23:30:00+00:00,1793143800,0.0,0.0,0.0,0.0


### Example - Period with Missing Values

In [45]:
df.head()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
1990-01-01 00:00:00+00:00,631152000,0.0,0.0,0.0,0.0
1990-01-01 00:30:00+00:00,631153800,0.0,0.0,0.0,0.0
1990-01-01 01:00:00+00:00,631155600,0.0,0.0,0.0,0.0
1990-01-01 01:30:00+00:00,631157400,0.0,0.0,0.0,0.0
1990-01-01 02:00:00+00:00,631159200,0.0,0.0,0.0,0.0


In [ ]:
df_melted = df.reset_index().melt(id_vars=["timestamp", "datetime"])
df_melted.head()

,timestamp,datetime,variable,value
0,631152000,1990-01-01 00:00:00+00:00,shortwave_radiation,0.0
1,631153800,1990-01-01 00:30:00+00:00,shortwave_radiation,0.0
2,631155600,1990-01-01 01:00:00+00:00,shortwave_radiation,0.0
3,631157400,1990-01-01 01:30:00+00:00,shortwave_radiation,0.0
4,631159200,1990-01-01 02:00:00+00:00,shortwave_radiation,0.0


In [55]:
missing_sample = df.loc["1990-01-25":"1990-02-01"]
px.line(
    missing_sample,
    x=missing_sample.index,
    y="shortwave_radiation",
    title="Example of period with missing data",
)

### Count by Time of the Day

In [75]:
fig = px.bar(
    df[df["shortwave_radiation"].isna()]
    .index.to_series()
    .dt.time.value_counts()
    .sort_index(),
    labels={"value": "Count", "datetime": "Time of the Day"},
    title="Count of Missing Values by Time of the Day",
)
fig.update_layout(showlegend=False)
fig.show()

Most of the missing values are during the midnight period (23:30 - 01:30).

### Missing Ranges

In [ ]:
missing_rows = df.loc[df.isna().any(axis=1)].reset_index()
range_id = (missing_rows["datetime"].diff() != pd.Timedelta("30min")).cumsum()

ranges = missing_rows.groupby(range_id).agg(
    start_ts=("timestamp", "first"),
    end_ts=("timestamp", "last"),
    start=("datetime", "first"),
    end=("datetime", "last"),
    steps=("datetime", "size"),
)

ranges["hours"] = ranges["steps"] / 2

ranges = ranges.reset_index(drop=True)
ranges

,start_ts,end_ts,start,end,steps,hours
0,631274400,631303200,1990-01-02 10:00:00+00:00,1990-01-02 18:00:00+00:00,17,8.5
1,633463200,633506400,1990-01-27 18:00:00+00:00,1990-01-28 06:00:00+00:00,25,12.5
2,633546000,633546000,1990-01-28 17:00:00+00:00,1990-01-28 17:00:00+00:00,1,0.5
3,633549600,633592800,1990-01-28 18:00:00+00:00,1990-01-29 06:00:00+00:00,25,12.5
4,633632400,633632400,1990-01-29 17:00:00+00:00,1990-01-29 17:00:00+00:00,1,0.5
...,...,...,...,...,...,...
840,1773532800,1773876600,2026-03-15 00:00:00+00:00,2026-03-18 23:30:00+00:00,192,96.0
841,1774915200,1774999800,2026-03-31 00:00:00+00:00,2026-03-31 23:30:00+00:00,48,24.0
842,1782864000,1782948600,2026-07-01 00:00:00+00:00,2026-07-01 23:30:00+00:00,48,24.0
843,1784505600,1784590200,2026-07-20 00:00:00+00:00,2026-07-20 23:30:00+00:00,48,24.0


#### Duration

In [18]:
pd.Series(
    {h: (ranges["hours"] <= h).mean() * 100 for h in [1, 2, 4, 8, 24, 100]}
).round(2)

1       71.36
2       92.07
4       93.96
8       95.03
24      98.70
100    100.00
dtype: float64

Most gaps (90%) last 2 hours or less, but about 5% run longer than 8 hours, likely due to major satellite outages.

## Conclusion

Most missing values occur at night, when there's no sun, so they could be filled with zeros.
Since they make up less than 0.5% of the data, no imputation is applied for now and rows with missing values are dropped for the ML model.
If a future algorithm requires complete data, we'll add imputation then.

# Basic Data Properties
## Overview

In [21]:
daily = df.resample("D")["shortwave_radiation"].mean().reset_index()

fig = px.line(
    daily,
    x="datetime",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "datetime": "Date"},
    title="Daily Shortwave Radiation Over Time",
)
end = daily["datetime"].max()
fig.update_layout(
    xaxis={
        "rangeslider_visible": True,
        "range": [end - pd.DateOffset(years=5), end],
    },
)
fig

## Shortwave Radiation by Months

In [93]:
daily_month = daily.assign(month=daily["datetime"].dt.month)

fig = px.box(
    daily_month,
    x="month",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "month": "Month"},
    title="Daily Mean Shortwave Radiation by Month",
)
fig.update_xaxes(
    type="category",
    tickvals=list(range(1, 13)),
    ticktext=MONTHS,
)
fig